## Structured output
#### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.
## Pydantic
#### Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")


In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description=" the title is movide")
    year:int = Field(description="This year the movei was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies ratings out of 10")

In [3]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x112c6f910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112d2fad0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': ' the title is movide', 'type': 'string'}, 'year': {'description': 'This year the movei was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10', 'type'

In [4]:
model_with_structure.invoke("provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

# Message output alonglide paresed structure 

In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description=" the title is movide")
    year:int = Field(description="This year the movei was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The movies ratings out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True )
model_with_structure

{
  raw: _ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x112c6f910>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x112d2fad0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': ' the title is movide', 'type': 'string'}, 'year': {'description': 'This year the movei was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies ratings out of 10

# Nested structure

In [6]:
from pydantic import BaseModel, Field

class Actor (BaseModel):
  name: str 
  role: str

class MovieDetails (BaseModel):
  title: str 
  year: int
  cast: list[Actor] 
  genres: list[str]
  budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(Movie, include_raw=True )

res = model_with_structure.invoke("Provide details about the movie Avengers")
res

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Avengers. Let me see which function I need to use. The available tool is the Movie function, which requires title, year, director, and rating.\n\nFirst, I need to figure out which specific Avengers movie they\'re referring to. There are several in the series. The original Avengers was released in 2012, directed by Joss Whedon. The latest one is The Marvels, but maybe they mean the first one. I\'ll assume the original unless specified otherwise.\n\nSo, the title would be "Avengers", year 2012, director Joss Whedon, and the rating. I should check the rating. It has an 8.1 on IMDb. Let me make sure all required parameters are included. Title, year, director, rating. Yep, that\'s all there. I\'ll structure the function call with these details.\n', 'tool_calls': [{'id': 'fmtq9kayt', 'function': {'arguments': '{"director":"Joss Whedon","rating":8.1,"title":"Avengers","y

# TypedDict

#### TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [7]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"] 
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float,..., "The movie's rating out of 10"]

model_withTypeDiect = model.with_structured_output(MovieDict)
res  = model_withTypeDiect.invoke("Please provide the deatils of the movie avengers")
res


{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [8]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

## Data classes

#### A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator w

In [11]:
## Data Classes. 
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """ Contact information for a person """
    name : str 
    email: str
    phone: str

agent = create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format = ContactInfo
)

result = agent.invoke({
    "messages":[{
        "role":"user",
        "content":"Extract contact info from: Jone Doe, john@example.com, (555) 123-4567"
    }]
}) 

result


{'messages': [HumanMessage(content='Extract contact info from: Jone Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9f34ec71-3086-476b-88b8-9570a48dede7'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'r1rmt7mga', 'function': {'arguments': '{"email":"john@example.com","name":"Jone Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 263, 'total_tokens': 297, 'completion_time': 0.074871127, 'completion_tokens_details': None, 'prompt_time': 0.030851441, 'prompt_tokens_details': None, 'queue_time': 0.051075759, 'total_time': 0.105722568}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e734e-d9dd-7ee1-ab2a-c9791da7403e-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'j